# Πείραμα 10 - iTransformer
#### Διπλωματική εργασία - Αναγνωστόπουλος Σπύρος

### 1) Περιγραφή πειράματος
Το iTransformer είναι μια παραλλαγή των μοντέλων Transformer. Εκμεταλλεύεται τον μηχανισμό self-attention για να εντοπίζει σχέσεις και εξαρτήσεις σε διαφορετικές χρονικές κλίμακες, κάτι που το καθιστά ιδιαίτερα αποτελεσματικό σε δεδομένα με μη γραμμικές δομές και μεγάλα χρονικά παράθυρα. Σε αντίθεση με τα παραδοσιακά RNN ή LSTM, ο iTransformer μπορεί να επεξεργαστεί όλες τις χρονικές στιγμές ταυτόχρονα, βελτιώνοντας έτσι την αποδοτικότητα και την ικανότητα σύλληψης μακροπρόθεσμων εξαρτήσεων.

Βασικά στοιχεία του iTransformer:

* Input embedding: μετατρέπει τις τιμές της χρονοσειράς σε διανύσματα κατάλληλα για το μοντέλο.
* Attention mechanism: εντοπίζει ποιες χρονικές στιγμές επηρεάζουν περισσότερο την τρέχουσα πρόβλεψη.
* Feed-forward layers: επεξεργάζονται τα ενδιάμεσα σήματα για να ενισχύσουν την εκμάθηση μη γραμμικών σχέσεων.
* Output projection: παράγει την τελική πρόβλεψη για τις μελλοντικές τιμές της χρονοσειράς.

#### Ερευνητικά Ερωτήματα Πειράματος 9 – iTransformer
- Σε σχέση με το τελικό ΤΤΜ, πού υπερτερεί ή υστερεί ο iTransformer στα σφάλματα και στην κατευθυντική ικανότητα;
- Ποια είναι η οικονομική αξία των προβλέψεων
- Μια στρατηγικη σχεδιασμένη με iTransformer μπορεί να αποφέρει περισσότερα κέρδη

### 2) Περιγραφή Μοντέλου & Εκπαίδευσης

**Στόχος πρόβλεψης**<br>
Το μοντέλο προβλέπει τον επόμενης ημέρας λογαριθμικό ημερήσιο αριθμοδείκτη απόδοσης (log-return) της τιμής κλεισίματος:

$$
r_t = \log(P_t) - \log(P_{t-1})
$$

Η μάθηση γίνεται στο χώρο των returns (όχι τιμών).

**Είσοδοι (features) & στόχος**<br>
Για κάθε ticker φορτώνεται χρονοσειρά με στήλες:

* Τιμές: open, high, low, close
* Τεχνικοί δείκτες: sma_20, ema_20, wma_20, tema_20, bb_upper, bb_lower

Στόχος (label) για το μοντέλο είναι το $r_t$ που υπολογίζεται από το close.

**Προεπεξεργασία & Διαχωρισμοί**<br>
* Υπολογισμός returns: από τη στήλη close υπολογίζονται τα $r_t$ (με `np.diff(log(close)) και ευθυγράμμιση με $r_0=0$).
* Χρονικοί διαχωρισμοί: με μήκος σειράς $N$,

  * Train region: τα πρώτα 42% δείγματα (train_end = int(0.42*N)).
  * (Η ύπαρξη test ορίζεται αργότερα στον κώδικα, αλλά η εκπαίδευση χρησιμοποιεί μόνο το 42%).
* Κλιμάκωση χαρακτηριστικών: MinMaxScaler εφαρμόζεται με fit μόνο στο train και κατόπιν transform σε όλο το εύρος (για συνεπή κλίμακα χωρίς leakage).
* Κλιμάκωση στόχου (returns): StandardScaler fit μόνο στο train και transform σε όλο το εύρος.
* Παράθυρα (sliding windows): χρησιμοποιείται σταθερό μήκος LOOKBACK = 256 ημερών.

  * Έγκυρα δείγματα ξεκινούν από index `lookback + 1` (απαιτείται $P_{t-1}$ για τον ορισμό του $r_t$).
  * Για κάθε χρόνο $i$: είσοδος $X_i \in \mathbb{R}^{256\times F}$ με τα τελευταία 256 βήματα και στόχος ο **scaled** return $r_i$.

Safe split: Οι scalers γίνονται fit μόνο στο train για αποφυγή look-ahead leakage.

**Αρχιτεκτονική iTransformer**<br>
Υλοποίηση σε PyTorch με encoder-only Transformer για μονοπρόβλεψη:

1. Input projection: Linear(n_features → d_model) για χαρτογράφηση των features στον κρυφό χώρο.
2. Διάταξη & Θέσεις: transpose σε σχήμα (seq, batch, d_model) και ημιτονοειδής positional encoding (σταθερή, μη-παραμετρική).
3. Transformer Encoder: στοίβα από num_layers = 2 στρώματα TransformerEncoderLayer με:

   * d_model = 128, nhead = 8, dim_feedforward = 256,
   * dropout = 0.1, ενεργοποίηση GELU, norm_first = True.
4. Συγκέντρωση: λαμβάνεται η αναπαράσταση του τελευταίου χρονικού βήματος.
5. Έξοδος (head): LayerNorm(d_model) -> Linear(d_model → 1) που δίνει έναν scalar, τον scaled $\hat r_t$.

* Είσοδος στο forward: x με σχήμα (batch, seq=256, features=F).
* Έξοδος: (batch,) προβλέψεις scaled returns.

**Διαμόρφωση Δεδομένων Εκπαίδευσης**<br>
Από τα κλιμακωμένα arrays δημιουργούνται tensors PyTorch:

* X_trn σχήματος (n_train_samples, 256, F)
* y_trn σχήματος (n_train_samples,)
* Validation split: τελευταίο 10% των training windows αποτελεί το validation set (χρονολογικά πρόσφατο τμήμα του train).

Δημιουργούνται DataLoader για train/val με:

* batch_size = min(64, len(X_trn))
* shuffle=True στο train, shuffle=False` στο val
* drop_last=False

**Καθεστώς Εκπαίδευσης**<br>
* Loss: MSE πάνω στα scaled returns.
* Βελτιστοποίηση: Adam με ρυθμό μάθησης LR = 1e-3.
* Gradient clipping:** κλιπάρισμα νόρμας στο 1.0 για σταθερότητα.
* EPOCHS = 30.
* Early stopping (χειροκίνητο):

  * Παρακολουθείται το validation MSE ανά epoch.
  * Διατηρείται το καλύτερο μοντέλο (αποθήκευση best_model.pt όταν το val loss βελτιώνεται πέρα από 1e-6).
  * Patience = 6: αν δεν υπάρχει βελτίωση για 6 συνεχόμενα epochs, γίνεται διακοπή.
  * Μετά το τέλος του loop, φορτώνονται τα καλύτερα βάρη για περαιτέρω χρήση.

**Αναπαραγωγιμότητα & Υποδομή**<br>
* Αυτόματη επιλογή συσκευής: cuda αν διαθέσιμη, αλλιώς cpu.
* Έλεγχοι ασφαλείας: αν το διαθέσιμο train δεν επαρκεί για το LOOKBACK, γίνεται RuntimeError.

**Διαστάσεις (ενδεικτικά)**<br>
* Αριθμός χαρακτηριστικών: $F = 9$ (όπως ορίζονται στα FEATURES).
* Παράθυρο: 256 χρονικά βήματα.
* Tensors train: (n_samples, 256, 9) -> forward σε (batch, 256, 9).


### 3) Οπτική αξιολόγηση iTransformer

Τα διαγράμματα του iTransformer δείχνουν πολύ σφιχτή επικάλυψη μεταξύ πρόβλεψης και πραγματικής τιμής στις περισσότερες μετοχές.

Στις AAPL, JNJ, PG η πρόβλεψη είναι σχεδόν επάνω στην τιμή, με πολύ μικρές αποκλίσεις γύρω από τα τοπικά άκρα.

Στις TSLA και AMD αναπαράγεται σωστά η μεσοπρόθεσμη δομή, αλλά τα ακραία άλματα αποτυπώνονται πιο ρηχά με τις κορυφές να είναι ελαφρά κατεσταλμένες.

Στην XOM έχουμε πολύ καλή στοίχιση στις ανοδικές και πτωτικές φάσεις του πετρελαϊκού κύκλου, αλλά μικρή χρονική υστέρηση σε απότομες διορθώσεις.

Στην SPY η ευρύτερη τάση συλλαμβάνεται καθαρά, αλλά ανά περιόδους η πρόβλεψη φαίνεται λίγο πιο λεία από την πραγματικότητα.

<div style="text-align: center;">
    <img src="./itransformer_result/testplots.png" alt="Chart" />
</div>

### 3)  Κατανομή σφάλματος πρόβλεψης σε σύγκριση με τo καλύτερο ΤΤΜ

Ο iTransformer έχει ελάχιστα βαρύτερες ουρές, ιδιαίτερα προς τα δεξιά (φαίνονται καποια outliers μέχρι ~+60), και ελαφρά δεξιά ασυμμετρία. Η κατανομή είναι συνολικά πιο πλατιά, αλλά έχουμε μεγαλύτερη πυκνότητα στο μηδέν.

Το τελικό μοντέλο έχει λιγότερα πολύ μεγάλα σφάλματα και είναι πιο συμμετρικό αλλά έχει τιμλες που αποκλίνουν από το μηδέν οπότε περιμένουμε να δούμε λίγο μικρότερες μετρικές στα σφάλματα για τον iTransformer.

<div style="text-align: center;">
    <img src="./itransformer_result/errors.png" alt="Chart" />
</div>

Στα σφάλματα, ο iTransformer τα πηγαίνει καλύτερα σε XOM και PG, όπου καταγράφονται τα χαμηλότερα MAE/RMSE (XOM ~0.93/1.29, PG ~1.07/1.53). Πολύ καλή είναι και η εικόνα της JNJ (MAE ~1.10, RMSE ~1.59). Στον SPY η σχετική απόδοση είναι επίσης ικανοποιητική, με την MAPE να πέφτει σε χαμηλά επίπεδα (~0.79), ένδειξη ότι το μοντέλο διαχειρίζεται καλά τη «μέση» δυναμική της αγοράς. Αντίθετα, οι πιο δύσκολες σειρές είναι η TSLA και η AMD· και στις δύο τα σφάλματα είναι αισθητά υψηλότερα (TSLA MAE ~6.23, RMSE ~9.18, AMD MAE ~2.16, RMSE ~3.20), κάτι αναμενόμενο για τίτλους υψηλού beta και απότομων καθεστώτων.

Στις κατευθυντικές μετρικές, οι καμπίνες συγκεντρώνονται γύρω από το 0.50, που σημαίνει πρακτικά «ελαφρώς καλύτερα από τυχαίο» αλλά με χρήσιμες διαφοροποιήσεις. Η JNJ εμφανίζει την πιο σταθερή κατευθυντική εικόνα (Hit Rate ~0.533, F1 ~0.535), ενώ ο SPY συνδυάζει αξιοπρεπή Precision/Recall (0.523/0.543) με F1 ~0.533, κάτι που ταιριάζει με τον χαμηλότερο θόρυβο του δείκτη. Η XOM έχει αυξημένο Hit Rate (~0.519) και ισορροπημένες Precision/Recall, δηλώνοντας καλή «σταθερότητα» πρόσημου. Η PG βρίσκεται επίσης λίγο πάνω από το 0.51–0.52 σε Precision/Recall. Η AMD είναι η ασθενέστερη περίπτωση (Precision ~0.464, Recall ~0.482, F1 ~0.473), ενώ η TSLA κυμαίνεται κοντά στο 0.50–0.51 με μικρή υπεροχή στο Recall.

Τα ραβδογράμματα επιβεβαιώνουν οπτικά τα παραπάνω: στο αριστερό (σφάλματα) η TSLA ξεχωρίζει με υψηλές μπάρες, ενώ XOM/PG έχουν τις χαμηλότερες. Στο δεξί (κατευθυντικά) όλες οι μπάρες κινούνται κοντά στο 0.5, με μικρές αλλά συνεπείς υπεροχές για JNJ, SPY και XOM.

$$
\begin{array}{lccccccc}
\textbf{Ticker} & \textbf{MAE} & \textbf{RMSE} & \textbf{MAPE} & \textbf{Hit Rate} & \textbf{Precision} & \textbf{Recall} & \textbf{F1} \\
\hline
AAPL & 1.722453 & 2.440332 & 1.341436 & 0.490675 & 0.512575 & 0.526445 & 0.519417 \\
TSLA & 6.234385 & 9.178932 & 3.209318 & 0.486379 & 0.498737 & 0.512322 & 0.505438 \\
XOM & 0.933726 & 1.285852 & 1.403276 & 0.518971 & 0.508951 & 0.522310 & 0.515544 \\
SPY & 2.926439 & 4.211686 & 0.797077 & 0.490675 & 0.523148 & 0.543269 & 0.533019 \\
JNJ & 1.096995 & 1.593847 & 0.784257 & 0.533119 & 0.527778 & 0.542857 & 0.535211 \\
AMD & 2.163509 & 3.201971 & 2.374952 & 0.479100 & 0.463602 & 0.482072 & 0.472656 \\
PG & 1.066117 & 1.532919 & 0.842830 & 0.490032 & 0.513158 & 0.526380 & 0.519685 \\
\end{array}
$$

<div style="text-align: center;">
    <img src="./itransformer_result/metrics.png" alt="Chart" />
</div>

###  Αποτελέσματα μετρικών ανά καθεστώς μεταβλητότητας

Ο πίνακας συνοψίζει την απόδοση του iTransformer αφού ομαδοποιήσαμε τις περιόδους του test set σε τέσσερα καθεστώτα. 

Στο καθεστώς **Υψηλής Μεταβλητότητας** τα σφάλματα κορυφώνονται (MAE≈4.20, RMSE≈6.19, MAPE≈2.79) και οι κατευθυντικές μετρικές υποχωρούν (Hit≈0.483, F1≈0.489). Οι απότομες, ασύμμετρες μεταβολές και οι χοντρές ουρές δημιουργούν έντονο distribution shift, με αποτέλεσμα το μοντέλο να “εξομαλύνει” την κίνηση και να χάνει τόσο επίπεδο όσο και πρόσημο.

Στο καθεστώς **Χαμηλής Μεταβλητότητας** το μοντέλο εμφανίζει τα μικρότερα σφάλματα (MAE≈1.08, RMSE≈1.56) και το υψηλότερο κατευθυντικό σήμα (Hit≈0.512, F1≈0.527). Η ομαλή δυναμική και η μεγαλύτερη προβλεψιμότητα των ροών τιμών ευνοούν την προσοχή/μάθηση του Transformer.

Στο καθεστώς **Αντιστροφής Τάσης** Οι επιδόσεις παραμένουν καλές και ισορροπημένες (MAE≈1.33, RMSE≈1.86, F1≈0.517). Οι μεταβάσεις από down σε up trend «διαβάζονται» επαρκώς, αν και η σχετική ακρίβεια (MAPE≈1.37) είναι χειρότερη από το low-vol, στοιχείο που υποδηλώνει μέτρια δυσκολία στην ταχύτητα προσαρμογής αμέσως μετά το γύρισμα.

Στο καθεστώς **Πλάγιας Ακατάστατης κίνησης** Παρότι το MAPE είναι το χαμηλότερο (≈0.80), τα απόλυτα σφάλματα είναι αυξημένα (MAE≈2.93, RMSE≈4.21). Οι τιμές βρίσκονται σε υψηλά επίπεδα (συνεπώς το απόλυτο λάθος μεγαλώνει) ενώ το μοντέλο συλλαμβάνει σχετικά καλά την αναλογική μεταβολή. Κατευθυντικά κινείται αξιοπρεπώς (**F1≈0.533**).


$$
\begin{array}{lccccccc}
\textbf{Regime Type} & \textbf{MAE} & \textbf{RMSE} & \textbf{MAPE} & \textbf{Hit Rate} & \textbf{Precision} & \textbf{Recall} & \textbf{F1} \\
\hline
High-Volatility & 4.198947 & 6.190451 & 2.792135 & 0.482739 & 0.481169 & 0.497197 & 0.489047 \\
Low-Volatility & 1.081556 & 1.563383 & 0.813544 & 0.511576 & 0.520468 & 0.534619 & 0.527448 \\
Post-Trend Reversal & 1.328089 & 1.863092 & 1.372356 & 0.504823 & 0.510763 & 0.524377 & 0.517481 \\
Sideways/Chop & 2.926439 & 4.211686 & 0.797077 & 0.490675 & 0.523148 & 0.543269 & 0.533019 \\
\end{array}
$$

### 6) Διαγράμματα κυλιόμενης μεταβλητότητας - σφάλματος

Το κρίσιμο εύρημα είναι ότι το νέφος του iTransformer (μπλε) πλαταίνει πιο έντονα από αυτό του FinalModel (πορτοκαλί), ιδίως στα ανώτερα δεκατημόρια της μεταβλητότητας. Σε χαμηλή μεταβλητότητα τα δύο μοντέλα συγκλίνουν.

Ανά μετοχή, η εικόνα είναι συνεκτική. Στην AAPL η σχέση μεταβλητότητας–σφάλματος είναι εμφανώς ανοδική, με τον iTransformer να παράγει περισσότερα μεγάλα λάθη για vol >0.05, ενώ το FinalModel παραμένει συμπαγές. Η TSLA παρουσιάζει το ισχυρότερο φαινόμενο. Στην υψηλή μεταβλητότητα συσσωρεύονται outliers σε επίπεδα σφάλματος 20–50, όταν το πορτοκαλί νέφος σπανίως ξεπερνά τα \~15. Στην XOM και στον SPY το πρότυπο είναι ηπιότερο αλλά ίδιο, δηλαδή ήπια κλίση σε χαμηλό vol και αισθητή διεύρυνση των λαθών του iTransformer καθώς το vol κινείται προς 0.04–0.08. Η JNJ και η PG, ως αμυντικές, έχουν συνολικά μικρότερες διασπορές και σχετικά αδύναμη κλίση. Ωστόσο ακόμη και εκεί, τα λίγα επεισόδια αυξημένου vol συνοδεύονται από δυσανάλογες εκρήξεις σφάλματος στον iTransformer. Η AMD προσομοιάζει την TSLA.


<div style="text-align: center;">
    <img src="./itransformer_result/rollvol.png" alt="Chart" />
</div>

### Pearson συσχέτιση (μεταβλητότητα ↔ σφάλμα) για iTransformer — ερμηνεία

Όλα τα tickers έχουν θετική συσχέτιση. Τα περισσότερα κινούνται στη ζώνη 0.20–0.38. Η PG ξεχωρίζει με 0.5654, ακολουθούν AAPL ≈ 0.38 και XOM ≈ 0.34, ενώ οι χαμηλότερες τιμές εμφανίζονται σε TSLA (0.21) και AMD (0.22).

**Σε σχέση με άλλα μοντέλα.**

* Έναντι **FinalModel**: ο iTransformer δείχνει εντονότερη σύζευξη με τη μεταβλητότητα σε AAPL, AMD, PG (μεγάλη διαφορά στην PG), αλλά ηπιότερη σε XOM, SPY, JNJ.
* Έναντι **ARIMA**: γενικά ασθενέστερη σύζευξη (εκτός της PG, όπου παραμένει υψηλή).
* Έναντι **LSTM**: το LSTM εμφάνιζε μηδενικές/αρνητικές συσχετίσεις. O iTransformer είναι σαφώς πιο ευαίσθητος στη μεταβλητότητα.



$$
\begin{array}{lccccccc}
\textbf{Ticker} & \textbf{Pearson correlation} \\
\hline
AAPL & 0.3780  \\
TSLA & 0.2114  \\
XOM & 0.3358  \\
SPY & 0.2804  \\
JNJ & 0.2588  \\
AMD & 0.2202  \\
PG & 0.5654  \\
\end{array}
$$

### Αποτελέσματα στα παράθυρα γεγονότων

**Κατάρρευση & Ανάκαμψη (AAPL, 2020-03-10).** Η ανοδική φάση πριν το σοκ αποδίδεται πιστά, το απότομο ξεπούλημα αναπαράγεται με μεγάλη  καθυστέρηση η οποία στη φάση της ανάκαμψης ακολουθείται μειώνεται.

**TSLA – High-Beta Cooling (Ιαν. 2021).** Το ανοδικό σκέλος αποτυπώνεται σωστά, με ήπια υποεκτίμηση. Η αποφόρτιση μετά την κορυφή συλλαμβάνεται επίσης σωστά, αλλά υπάρχει καθυστέρηση κάποιων ημερών στην τιμή και γενικά τα μικρά σκαμπανεβάσματα δεν αποτυπώνονται σωστά.

**XOM – Oil Cycle Peak (Ιούν. 2022).** Η κλιμακωτή άνοδος του κύκλου πετρελαίου παρακολουθείται πάρα πολύ καλά. Το σημείο κορύφωσης/στροφής καταγράφεται με μικρή καθυστέρηση και τα επόμενα κυματικά μοτίβα αναπαράγονται με καλή χρονική στοίχιση. Είναι η καλύτερη απόδοση για το παράθυρο της XOM που έχουμε δει μέχρι τώρα.

**SPY – Drawdown Chop (Φθινόπωρο 2022).** Αν και η γενική κατεύθυνση είναι σωστή ο iTransformer έχει κάνει πάρα πολυ μεγάλη εξομάλυνση της τιμής, και λόγω αυτού θα είναι λογικό οι κατευθυντικές μετρικές να είναι σε πολύ χαμηλά επίπεδα.

**JNJ – Low-Volatility Stretch (2019).** Σε περιβάλλον χαμηλής μεταβλητότητας η πρόβλεψη κάθεται πρακτικά πάνω στην τιμή, με αμελητέες αποκλίσεις ακόμη και γύρω από βραχείες ανωμαλίες.

**AMD – Tech Selloff (Οκτ. 2018).** Αρκετά παρόμοιο αποτέλεσμα με αυτό της TSLA.

**PG – Macro-Irrelevant Calm (2015).** Εξαιρετικά στενό ταίριασμα σε όλο το παράθυρο.

<div style="text-align: center;">
    <img src="./itransformer_result/eventplots.png" alt="Chart" />
</div>

### iTransformer — απόδοση σε «γεγονότα» (event windows)

Στο επίπεδο σφάλματος, ο iTransformer βελτιώνεται καθαρά σε AAPL, JNJ, AMD, PG (λ.χ. AAPL: MAE ≈ 1.14 από \~1.72, JNJ: \~0.86 από \~1.10, PG: \~0.42 από \~1.07), ενώ χειροτερεύει σε SPY, TSLA, XOM (SPY: MAE \~4.57 από \~2.93· TSLA: \~6.40 από \~6.23· XOM: \~1.20 από \~0.93). Η εικόνα του MAPE βοηθά να ερμηνευτούν αντιφάσεις. Για την AMD το MAE είναι πολύ χαμηλό επειδή το επίπεδο τιμών στο sell-off του 2018 ήταν μικρό, αλλά το MAPE ανεβαίνει (\~2.99%). Αντίθετα, στον SPY το MAPE αυξάνεται απότομα (\~1.17% από \~0.79%), στοιχείο που δείχνει ότι το drawdown-chop κάνει δύσκολη και τη σχετική και την απόλυτη πρόβλεψη.

Ως προς την κατεύθυνση, ο iTransformer αποδίδει καλύτερα σε περιβάλλοντα με πιο καθαρή δυναμική. XOM και JNJ καταγράφουν F1 ≈ 0.57 και 0.55, με ταυτόχρονη άνοδο τόσο του precision όσο και του recall έναντι του test set, ενώ ο PG έχει πολύ καλο hit-rate ≈ 0.51. Ο SPY στο εμφανίζει σημαντική πτώση κατευθυντικών μετρικών (precision ≈ 0.41, F1 ≈ 0.41). Στην TSLA τα σφάλματα είναι υψηλά και η F1 κινείται οριακά πάνω στο 0.46. Η AAPL βελτιώνει τα σφάλματα στη φάση, αλλά οι κατευθυντικές μετρικές μένουν γύρω στο 0.47.

Συνοπτικά, στα παράθυρα γεγονότων ο iTransformer κρατά ή και βελτιώνει την ακρίβεια επιπέδου όταν η δυναμική είναι πιο ομαλή ή κυκλική, αλλά γενικώς έχει αστθή αποτελέσματα



$$
\begin{array}{lccccccc}
\textbf{Ticker} & \textbf{MAE} & \textbf{RMSE} & \textbf{MAPE} & \textbf{Hit Rate} & \textbf{Precision} & \textbf{Recall} & \textbf{F1} \\
\hline
AAPL & 1.139225 & 1.771298 & 1.732378 & 0.425641 & 0.466667 & 0.466667 & 0.466667 \\
TSLA & 6.399224 & 8.529700 & 3.476710 & 0.430769 & 0.456522 & 0.463235 & 0.459854 \\
XOM & 1.196707 & 1.608741 & 1.610268 & 0.526923 & 0.564626 & 0.584507 & 0.574394 \\
SPY & 4.574285 & 5.855461 & 1.174302 & 0.501931 & 0.405405 & 0.416667 & 0.410959 \\
JNJ & 0.862552 & 1.401040 & 0.757509 & 0.513158 & 0.541667 & 0.561728 & 0.551515 \\
AMD & 0.608916 & 0.911287 & 2.986436 & 0.447876 & 0.467153 & 0.477612 & 0.472325 \\
PG & 0.423232 & 0.584479 & 0.701945 & 0.514085 & 0.466667 & 0.488372 & 0.477273 \\
\end{array}
$$

Το γράφημα απεικονίζει την εξέλιξη του σωρευτικού κέρδους της στρατηγικής (σε ποσοστό) για κάθε μετοχή στο χρόνο. Η TSLA έχει μεγάλα ανοδικά σκέλη και έντονες πτώσεις, αλλά κλείνει υψηλά (\~110–130%), στοιχείο υψηλής απόδοσης με υψηλή μεταβλητότητα. Η AMD παρουσιάζει επίσης ισχυρές εκρηκτικές φάσεις (κορύφωση >100%) με ενδιάμεσες πτώσεις. Ο SPY ακολουθεί ηπιότερη, πιο ομαλή ανιούσα τροχιά που τελικά σταθεροποιείται γύρω στο 60–70%, υποδηλώνοντας καλύτερο λόγο απόδοσης/κινδύνου. Η PG κινείται σταθερά με μικρά εύρη και μέτριο τελικό κέρδος (\~20–30%), ενώ η JNJ και η XOM καταγράφουν αρχικά αξιοπρόσεκτα κέρδη που αμβλύνονται στο δεύτερο μισό της περιόδου. Η AAPL είναι η πιο αδύναμη καμπύλη, με παρατεταμένες φάσεις υποχώρησης και αρνητικό κλείσιμο. 

<div style="text-align: center;">
    <img src="./itransformer_result/strategy_cumulative_returns_last40.png" alt="Chart" />
</div>

### Αξιολόγηση οικονομικών με iTransformer (Sharpe – Cumulative Return – Max Drawdown)


**Κύρια ευρήματα ανά δείκτη.**

* **Sharpe ratio.** Η καλύτερη αναλογία απόδοσης/κινδύνου εμφανίζεται στον SPY (≈0.04), ένδειξη ότι οι προβλέψεις του iTransformer μεταφράζονται σε σχετικά καθαρές κινήσεις στην αγορά. Ακολουθούν AMD και TSLA (≈0.02–0.025), έπειτα PG (\~0.01), ενώ XOM** και JNJ κινούνται οριακά θετικά. Η AAPL είναι ουσιαστικά μηδενική/ελαφρά αρνητική. Συνεπώς, από πλευράς απόδοσης το μοντέλο λειτουργεί πιο αξιόπιστα σε index/mega-caps (SPY) και σε ορισμένα high-beta (AMD/TSLA) όταν γίνει σωστή διαχείριση κινδύνου.

* **Cumulative return.** Οι υψηλότερες σωρευτικές αποδόσεις προκύπτουν σε TSLA (\~112%) και AMD (\~111%), ακολουθεί το SPY (\~71%). PG (\~22%) και XOM (\~11%) δίνουν μέτριες αποδόσεις, το JNJ χαμηλή (~6%), ενώ η AAPL είναι αρνητική (~~−3%). Το μοτίβο υποδεικνύει ότι ο iTransformer κεφαλαιοποιεί καλά τις παρατεταμένες τάσεις σε high-beta ονόματα και στον ευρύτερο δείκτη, αλλά δυσκολεύεται σε πιο αμυντικά περιβάλλοντα.

* **Max drawdown.** Το κόστος ρίσκου είναι έντονο σε TSLA και AMD (πολύ υψηλά drawdowns), αυξημένο και στην XOM, πιο ήπιο στον SPY (η μικρότερη μπάρα μεταξύ των υψηλών αποδόσεων), χαμηλό στην PG και μέτριο στην JNJ. Με απλά λόγια, εκεί όπου κερδίζουμε πολλά (TSLA/AMD) πληρώνουμε μεγάλα ζημιών αν δεν υπάρχει έλεγχος θέσης.



<div style="text-align: center;">
    <img src="./itransformer_result/econ_barplots_last40.png" alt="Chart" />
</div>

**1) Θερμικοί χάρτες σφαλμάτων**<br>
Οι τιμές είναι αρνητικές σχεδόν παντού, άρα ο FinalModel εμφανίζει μεγαλύτερα σφάλματα από το iTransformer. Η υστέρηση είναι εντονότερη στα TSLA και SPY (π.χ. RMSE ≈ −0.95 και −1.19 αντίστοιχα, με αντίστοιχες απώλειες και σε MAE/MAPE), σημαντική στα AMD και AAPL (RMSE ≈ −0.63 και −0.42), και μικρότερη αλλά σταθερή στα XOM, JNJ και PG. 

**2) Θερμικοί χάρτες κατευθυνσης**<br>
Oι θετικές τιμές σημαίνουν ότι το τελικό μοντέλο έχει καλύτερη κατεύθυνση (hit/precision/recall/F1) από τον iTransformer, ενώ οι αρνητικές ότι υστερεί. Παρατηρούμε καθαρό πλεονέκτημα του τελικού μοντέλου στα TSLA και AMD (ιδίως στο recall και το F1 με βελτιώσεις ~0.02–0.03 μονάδες), σαφή υπεροχή του iTransformer στα XOM και JNJ (αρνητικές διαφορές σε όλα τα directional), και πιο μικτές εικόνες στα AAPL, SPY και PG, όπου οι διαφορές είναι μικρές και εναλλάσσονται ανά δείκτη.

**3) Wilcoxon**
Ο έλεγχος Wilcoxon (two-sided) δείχνει ότι οι διαφορές στα σφάλματα είναι στατιστικά σημαντικές (MAE/RMSE/MAPE: p = 0.015625 < 0.05), οπότε απορρίπτεται η ισότητα των μοντέλων στις μετρικές λάθους. Αντίθετα, για τα κατευθυντικά μεγέθη (hit rate, precision, recall, F1) οι p-values είναι υψηλές (0.4688, 0.3750, 0.2969, 0.9375), άρα δεν τεκμηριώνεται συνολικά στατιστική διαφορά.



<div style="text-align: center;">
    <img src="./itransformer_result/heatmap_error_improvement_iTransformer_minus_Final.png" alt="Chart" width='700' />
</div>

<div style="text-align: center;">
    <img src="./itransformer_result/heatmap_directional_delta_Final_minus_iTransformer.png" alt="Chart" width='700'/>
</div>

$$
\begin{array}{lccccccc}
\textbf{Ticker} & \textbf{Wilc} \\
\hline
mae & 0.015625 \\
rmse & 0.015625 \\
mape & 0.015625 \\
hit\_rate & 0.468750 \\
precision & 0.375000 \\
recall & 0.296875 \\
f1 & 0.937500 \\
\hline
\end{array}
$$
